In [ ]:
"""
The most atomic way to train and run inference for a GPT in pure, dependency-free Python.
This file is the complete algorithm.
Everything else is just efficiency.

Extended with GELU, LoRA, RoPE, and a tiny Mixture-of-Experts block.

@karpathy
"""

import os       # os.path.exists
import math     # math.log, math.exp, math.tanh, math.cos, math.sin
import random   # random.seed, random.choices, random.gauss, random.shuffle
random.seed(42) # Let there be order among chaos

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
USE_GELU = True
USE_ROPE = True
USE_LORA = False      # LoRA is most useful for fine-tuning a pretrained model
USE_MOE = True

NUM_EXPERTS = 3
TOP_K_EXPERTS = 2
LORA_RANK = 4
LORA_ALPHA = 8.0
ROPE_THETA = 10000.0

# Let there be a Dataset `docs`: list[str] of documents (e.g. a list of names)
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)
print(f"num docs: {len(docs)}")

# Let there be a Tokenizer to translate strings to sequences of integers ("tokens") and back
uchars = sorted(set(''.join(docs))) # unique characters in the dataset become token ids 0..n-1
BOS = len(uchars) # token id for a special Beginning of Sequence (BOS) token
vocab_size = len(uchars) + 1 # total number of unique tokens, +1 is for BOS
print(f"vocab size: {vocab_size}")

# Let there be Autograd to recursively apply the chain rule through a computation graph
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads') # Python optimization for memory usage

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # scalar value of this node calculated during forward pass
        self.grad = 0                   # derivative of the loss w.r.t. this node, calculated in backward pass
        self._children = children       # children of this node in the computation graph
        self._local_grads = local_grads # local derivative of this node w.r.t. its children

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other):
        return Value(self.data**other, (self,), (other * self.data**(other-1),))

    def log(self):
        return Value(math.log(self.data), (self,), (1/self.data,))

    def exp(self):
        return Value(math.exp(self.data), (self,), (math.exp(self.data),))

    def tanh(self):
        t = math.tanh(self.data)
        return Value(t, (self,), (1 - t * t,))

    def relu(self):
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

# Initialize the parameters, to store the knowledge of the model
n_layer = 1     # depth of the transformer neural network (number of layers)
n_embd = 16     # width of the network (embedding dimension)
block_size = 16 # maximum context length of the attention window (note: the longest name is 15 characters)
n_head = 4      # number of attention heads
head_dim = n_embd // n_head # derived dimension of each head
assert head_dim % 2 == 0, 'RoPE needs an even head dimension'

matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {}
trainable_keys = []


def register_matrix(name, nout, nin, std=0.08, trainable=True):
    state_dict[name] = matrix(nout, nin, std)
    if trainable:
        trainable_keys.append(name)


def register_linear(name, nout, nin, *, use_lora=False, std=0.08):
    # Store a frozen base matrix plus optional trainable low-rank adapters.
    register_matrix(name, nout, nin, std=std, trainable=not use_lora)
    if use_lora:
        register_matrix(f'{name}.lora_a', LORA_RANK, nin, std=0.02, trainable=True)
        register_matrix(f'{name}.lora_b', nout, LORA_RANK, std=0.0, trainable=True)


register_matrix('wte', vocab_size, n_embd)
if not USE_ROPE:
    register_matrix('wpe', block_size, n_embd)
register_matrix('lm_head', vocab_size, n_embd)
for i in range(n_layer):
    for proj in ('attn_wq', 'attn_wk', 'attn_wv', 'attn_wo'):
        register_linear(f'layer{i}.{proj}', n_embd, n_embd, use_lora=USE_LORA)
    if USE_MOE:
        register_matrix(f'layer{i}.moe_gate', NUM_EXPERTS, n_embd)
        for expert_idx in range(NUM_EXPERTS):
            register_matrix(f'layer{i}.expert{expert_idx}.fc1', 4 * n_embd, n_embd)
            register_matrix(f'layer{i}.expert{expert_idx}.fc2', n_embd, 4 * n_embd)
    else:
        register_matrix(f'layer{i}.mlp_fc1', 4 * n_embd, n_embd)
        register_matrix(f'layer{i}.mlp_fc2', n_embd, 4 * n_embd)
params = [p for key in trainable_keys for row in state_dict[key] for p in row]
print(f"num params: {len(params)}")

# Define the model architecture: a function mapping tokens and parameters to logits over what comes next
# Follow GPT-2, blessed among the GPTs, with minor differences: layernorm -> rmsnorm, optional RoPE,
# optional LoRA adapters, and a tiny MoE feed-forward block.
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]


def linear_with_lora(x, name):
    y = linear(x, state_dict[name])
    if USE_LORA:
        low_rank = linear(x, state_dict[f'{name}.lora_a'])
        delta = linear(low_rank, state_dict[f'{name}.lora_b'])
        scale = LORA_ALPHA / LORA_RANK
        y = [yi + scale * di for yi, di in zip(y, delta)]
    return y


def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]


def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]


def gelu(x):
    c = math.sqrt(2.0 / math.pi)
    return 0.5 * x * (1.0 + (c * (x + 0.044715 * (x ** 3))).tanh())


def apply_rope(x, pos_id):
    rotated = []
    for i in range(0, len(x), 2):
        angle = pos_id / (ROPE_THETA ** (i / len(x)))
        cos_a = math.cos(angle)
        sin_a = math.sin(angle)
        x0, x1 = x[i], x[i + 1]
        rotated.append(x0 * cos_a - x1 * sin_a)
        rotated.append(x0 * sin_a + x1 * cos_a)
    return rotated


def apply_rope_by_head(x, pos_id):
    out = []
    for h in range(n_head):
        hs = h * head_dim
        out.extend(apply_rope(x[hs:hs + head_dim], pos_id))
    return out


def topk_indices(values, k):
    return sorted(range(len(values)), key=lambda idx: values[idx].data, reverse=True)[:k]


def dense_mlp(x, li):
    x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
    x = [gelu(xi) if USE_GELU else xi.relu() for xi in x]
    return linear(x, state_dict[f'layer{li}.mlp_fc2'])


def moe_mlp(x, li):
    gate_probs = softmax(linear(x, state_dict[f'layer{li}.moe_gate']))
    expert_ids = topk_indices(gate_probs, TOP_K_EXPERTS)
    selected_total = sum(gate_probs[idx] for idx in expert_ids)
    out = [Value(0.0) for _ in range(n_embd)]
    for expert_idx in expert_ids:
        hidden = linear(x, state_dict[f'layer{li}.expert{expert_idx}.fc1'])
        hidden = [gelu(xi) if USE_GELU else xi.relu() for xi in hidden]
        expert_out = linear(hidden, state_dict[f'layer{li}.expert{expert_idx}.fc2'])
        weight = gate_probs[expert_idx] / selected_total
        out = [oi + weight * ei for oi, ei in zip(out, expert_out)]
    return out


def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id] # token embedding
    if USE_ROPE:
        x = list(tok_emb)
    else:
        pos_emb = state_dict['wpe'][pos_id] # learned absolute position embedding
        x = [t + p for t, p in zip(tok_emb, pos_emb)] # joint token and position embedding
    x = rmsnorm(x) # note: not redundant due to backward pass via the residual connection

    for li in range(n_layer):
        # 1) Multi-head Attention block
        x_residual = x
        x = rmsnorm(x)
        q = linear_with_lora(x, f'layer{li}.attn_wq')
        k = linear_with_lora(x, f'layer{li}.attn_wk')
        v = linear_with_lora(x, f'layer{li}.attn_wv')
        if USE_ROPE:
            q = apply_rope_by_head(q, pos_id)
            k = apply_rope_by_head(k, pos_id)
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear_with_lora(x_attn, f'layer{li}.attn_wo')
        x = [a + b for a, b in zip(x, x_residual)]

        # 2) Feed-forward block: either dense MLP or sparse MoE
        x_residual = x
        x = rmsnorm(x)
        x_ffn = moe_mlp(x, li) if USE_MOE else dense_mlp(x, li)
        x = [a + b for a, b in zip(x_ffn, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

# Let there be Adam, the blessed optimizer and its buffers
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params) # first moment buffer
v = [0.0] * len(params) # second moment buffer

# Repeat in sequence
num_steps = 1000 # number of training steps
for step in range(num_steps):

    # Take single document, tokenize it, surround it with BOS special token on both sides
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # Forward the token sequence through the model, building up the computation graph all the way to the loss
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses) # final average loss over the document sequence. May yours be low.

    # Backward the loss, calculating the gradients with respect to all model parameters
    loss.backward()

    # Adam optimizer update: update the model parameters based on the corresponding gradients
    lr_t = learning_rate * (1 - step / num_steps) # linear learning rate decay
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end='\r')

# Inference: may the model babble back to us
temperature = 0.5 # in (0, 1], control the "creativity" of generated text, low to high
print("\n--- inference (new, hallucinated names) ---")
for sample_idx in range(20):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")




num docs: 32033
vocab size: 27
num params: 4192
step 1000 / 1000 | loss 2.6497
--- inference (new, hallucinated names) ---
sample  1: kamon
sample  2: ann
sample  3: karai
sample  4: jaire
sample  5: vialan
sample  6: karia
sample  7: yeran
sample  8: anna
sample  9: areli
sample 10: kaina
sample 11: konna
sample 12: keylen
sample 13: liole
sample 14: alerin
sample 15: earan
sample 16: lenne
sample 17: kana
sample 18: lara
sample 19: alela
sample 20: anton
